# Lumen Agentic Labelling Walkthrough

This notebook demonstrates the end-to-end labelling loop:

1. **Bootstrap** a Label Studio project
2. **Prelabel** images with model predictions
3. **Simulate human review** (accept/correct)
4. **Retrain** incrementally from corrections
5. **Evaluate** quality improvement
6. **Promote** through the model registry

### Prerequisites

```bash
pip install lumen[labelstudio]
docker compose up -d  # Start Label Studio
```

In [ ]:
from pathlib import Path

from lumen.annotation import LabelStudioClient, LabelStudioConfig, LabellingTaskStore
from lumen.annotation.review_loop import ReviewLoop, ReviewLoopConfig

## 1. Connect to Label Studio

In [ ]:
LS_URL = "http://localhost:8080"
LS_API_KEY = "your-api-key"  # Get from Account & Settings

client = LabelStudioClient(url=LS_URL, api_key=LS_API_KEY)
print(f"Label Studio health: {client.health()}")

## 2. Bootstrap a Label Studio Project

In [ ]:
project_id = client.bootstrap_project(
    name="LiveCELL Segmentation v1",
    task_type="segmentation",
    class_names=("cell",),
)
print(f"Project ID: {project_id}")

## 3. Prelabel: Push Model Predictions

In [ ]:
from unittest.mock import MagicMock

# Example: push images with synthetic predictions
image_paths = ["/data/livecell/cell_001.png", "/data/livecell/cell_002.png"]

# Build prediction objects (in practice, these come from MicroscopyInference)
predictions = {
    image_paths[0]: [
        MagicMock(
            class_name="cell",
            confidence=0.92,
            points=[(10, 10), (50, 10), (50, 50), (10, 50)],
        ),
    ],
}

config = LabelStudioConfig(task_type="segmentation", class_names=("cell",))
result = client.push_tasks(
    project_id,
    image_paths,
    predictions_by_image=predictions,
    config=config,
)
print(f"Pushed {len(result)} tasks")

## 4. Simulate Human Review

In a real workflow, human annotators review and correct predictions in the Label Studio UI.
Here we programmatically accept tasks to simulate the review step.

In [ ]:
# Pull annotations and accept all tasks
annotations = client.pull_annotations(project_id)
task_ids = [a["id"] for a in annotations]
client.mark_reviewed(task_ids)
print(f"Marked {len(task_ids)} tasks as reviewed")

## 5. Pull Corrections via ReviewLoop

In [ ]:
loop = ReviewLoop(
    client,
    ReviewLoopConfig(
        label_config=LabelStudioConfig(task_type="segmentation", class_names=("cell",)),
        output_root=Path("data/review-loop"),
        train_ratio=0.8,
        val_ratio=0.1,
    ),
)

corrected = loop.pull(project_id=project_id)
print(f"Train: {len(corrected.train)}, Val: {len(corrected.val)}, Holdout: {len(corrected.holdout)}")

## 6. Retrain from Corrections

In [ ]:
from lumen.retrain import IncrementalRetrainer, ModelRegistry, RetrainConfig

registry = ModelRegistry("weights/registry")

# Define a trainer factory (adapts to your model architecture)
def make_trainer(ckpt_path: str):
    # Load your segmentation trainer from the checkpoint
    pass

retrainer = IncrementalRetrainer(make_trainer, registry=registry)

report = retrainer.run(
    base_ckpt="eupe-livecell@latest",
    corrected_dataset=corrected,
    config=RetrainConfig(epochs=5, replay_buffer_ratio=0.2),
    promote_alias="eupe-livecell@latest",
    promote_gate="miou_delta>=+0.005",
)
print(f"Promoted: {report.promoted}, mIoU: {report.metrics.get('miou', 0):.4f}")

## 7. Inspect Model Registry

In [ ]:
for entry in registry.list():
    print(f"{entry['alias']}: {entry['ckpt']}")
    print(f"  Metrics: {entry.get('metrics', {})}")